In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [7]:
# Importing required libraries
import sqlite3
import numpy as np
import pandas as pd
import seaborn as sns
import pandasql as ps
import matplotlib.pyplot as plt
import os
from datetime import datetime
from IPython.core.magic import register_line_magic
from sqlalchemy import create_engine
import sqlalchemy

In [4]:
#!pip install jupysql

### SQLITE CONNECTION FUNCTION

In [9]:
def sqliteConnections():
    try:        
        sqlitedb_path = os.getenv("SQLITE_DB", "travel.sqlite")
        with sqlite3.connect(sqlitedb_path) as sqliteConn:
            engine = create_engine(f"sqlite:////{sqlitedb_path}", echo=True) 
            print("✅ Connection Successful")
            return sqliteConn, engine  
        
    except sqlite3.Error as e:
        print(f"❌ Error Connecting to .db: {e}")
        return None, None

In [10]:
# CONNECTIONS
sqlite_conn, engine = sqliteConnections()

✅ Connection Successful


In [3]:
%load_ext sql

In [11]:
%sql sqlite:////kaggle/input/airlines-dataset/travel.sqlite

Connecting to 'sqlite:////kaggle/input/airlines-dataset/travel.sqlite'

In [ ]:
#sqlite_conn=sqlite3.connect("/kaggle/input/airlines-dataset/travel.sqlite")

### PROGRAMMATIC ASSESSMENT

- SQL tables

In [12]:
%%sql
SELECT name FROM sqlite_schema WHERE type ='table';

Running query in 'sqlite:////kaggle/input/airlines-dataset/travel.sqlite'

name
aircrafts_data
airports_data
boarding_passes
bookings
flights
seats
ticket_flights
tickets


#### Aircrafts Dataset

In [13]:
%%sql 
SELECT * 
FROM aircrafts_data 
LIMIT 10;

Running query in 'sqlite:////kaggle/input/airlines-dataset/travel.sqlite'

aircraft_code,model,range
773,"{""en"": ""Boeing 777-300"", ""ru"": ""Боинг 777-300""}",11100
763,"{""en"": ""Boeing 767-300"", ""ru"": ""Боинг 767-300""}",7900
SU9,"{""en"": ""Sukhoi Superjet-100"", ""ru"": ""Сухой Суперджет-100""}",3000
320,"{""en"": ""Airbus A320-200"", ""ru"": ""Аэробус A320-200""}",5700
321,"{""en"": ""Airbus A321-200"", ""ru"": ""Аэробус A321-200""}",5600
319,"{""en"": ""Airbus A319-100"", ""ru"": ""Аэробус A319-100""}",6700
733,"{""en"": ""Boeing 737-300"", ""ru"": ""Боинг 737-300""}",4200
CN1,"{""en"": ""Cessna 208 Caravan"", ""ru"": ""Сессна 208 Караван""}",1200
CR2,"{""en"": ""Bombardier CRJ-200"", ""ru"": ""Бомбардье CRJ-200""}",2700


- Extracting the english name from model

In [14]:
%%sql
SELECT aircraft_code, 
    json_extract(model, '$.en') AS aircraft_model, 
    range
FROM aircrafts_data
LIMIT 10;

Running query in 'sqlite:////kaggle/input/airlines-dataset/travel.sqlite'

aircraft_code,aircraft_model,range
773,Boeing 777-300,11100
763,Boeing 767-300,7900
SU9,Sukhoi Superjet-100,3000
320,Airbus A320-200,5700
321,Airbus A321-200,5600
319,Airbus A319-100,6700
733,Boeing 737-300,4200
CN1,Cessna 208 Caravan,1200
CR2,Bombardier CRJ-200,2700


#### Airports Dataset
- Extracting english airport name and city name

In [15]:
%%sql
SELECT airport_code, 
        json_extract(airport_name, '$.en') AS airport_name, 
        json_extract(city, '$.en') AS city, 
        coordinates, timezone
FROM airports_data 
LIMIT 5;

Running query in 'sqlite:////kaggle/input/airlines-dataset/travel.sqlite'

airport_code,airport_name,city,coordinates,timezone
YKS,Yakutsk Airport,Yakutsk,"(129.77099609375,62.0932998657226562)",Asia/Yakutsk
MJZ,Mirny Airport,Mirnyj,"(114.03900146484375,62.534698486328125)",Asia/Yakutsk
KHV,Khabarovsk-Novy Airport,Khabarovsk,"(135.18800354004,48.5279998779300001)",Asia/Vladivostok
PKC,Yelizovo Airport,Petropavlovsk,"(158.453994750976562,53.1679000854492188)",Asia/Kamchatka
UUS,Yuzhno-Sakhalinsk Airport,Yuzhno-Sakhalinsk,"(142.718002319335938,46.8886985778808594)",Asia/Sakhalin


#### Boarding-passes Dataset

In [16]:
%%sql
# load boarding-passes data
SELECT *
FROM boarding_passes 
LIMIT 5;

Running query in 'sqlite:////kaggle/input/airlines-dataset/travel.sqlite'

ticket_no,flight_id,boarding_no,seat_no
0005435212351,30625,1,2D
0005435212386,30625,2,3G
0005435212381,30625,3,4H
0005432211370,30625,4,5D
0005435212357,30625,5,11A


- Checking duplicates

In [17]:
%%sql
# check for duplicate ticket numbers
SELECT 
        COUNT(ticket_no) AS total, 
        COUNT(DISTINCT ticket_no) AS unique_tickets,
        COUNT(ticket_no) - COUNT(DISTINCT ticket_no) AS dup_tickets
FROM boarding_passes;

Running query in 'sqlite:////kaggle/input/airlines-dataset/travel.sqlite'

total,unique_tickets,dup_tickets
579686,238834,340852


#### Bookings Dataset

In [18]:
%%sql
# load bookings data
SELECT *
FROM bookings 
LIMIT 5;

Running query in 'sqlite:////kaggle/input/airlines-dataset/travel.sqlite'

book_ref,book_date,total_amount
00000F,2017-07-05 03:12:00+03,265700
000012,2017-07-14 09:02:00+03,37900
000068,2017-08-15 14:27:00+03,18100
000181,2017-08-10 13:28:00+03,131800
0002D8,2017-08-07 21:40:00+03,23600


- Checking duplicates

In [19]:
%%sql
# check for duplicate bookings references
SELECT 
        COUNT(book_ref) AS total, 
        COUNT(DISTINCT book_ref) unique_ref,
        COUNT(book_ref) - COUNT(DISTINCT book_ref) AS dup_ref
FROM bookings;

Running query in 'sqlite:////kaggle/input/airlines-dataset/travel.sqlite'

total,unique_ref,dup_ref
262788,262788,0


#### Flights Dataset

In [20]:
%%sql
# load flights data
SELECT *
FROM flights 
LIMIT 5;

Running query in 'sqlite:////kaggle/input/airlines-dataset/travel.sqlite'

flight_id,flight_no,scheduled_departure,scheduled_arrival,departure_airport,arrival_airport,status,aircraft_code,actual_departure,actual_arrival
1185,PG0134,2017-09-10 09:50:00+03,2017-09-10 14:55:00+03,DME,BTK,Scheduled,319,\N,\N
3979,PG0052,2017-08-25 14:50:00+03,2017-08-25 17:35:00+03,VKO,HMA,Scheduled,CR2,\N,\N
4739,PG0561,2017-09-05 12:30:00+03,2017-09-05 14:15:00+03,VKO,AER,Scheduled,763,\N,\N
5502,PG0529,2017-09-12 09:50:00+03,2017-09-12 11:20:00+03,SVO,UFA,Scheduled,763,\N,\N
6938,PG0461,2017-09-04 12:25:00+03,2017-09-04 13:20:00+03,SVO,ULV,Scheduled,SU9,\N,\N


- Filtering missing data

In [21]:
%%sql
# load flights data
SELECT *
FROM flights
WHERE actual_departure != "\N" OR actual_arrival != "\N"
LIMIT 5;

Running query in 'sqlite:////kaggle/input/airlines-dataset/travel.sqlite'

flight_id,flight_no,scheduled_departure,scheduled_arrival,departure_airport,arrival_airport,status,aircraft_code,actual_departure,actual_arrival
1,PG0405,2017-07-16 09:35:00+03,2017-07-16 10:30:00+03,DME,LED,Arrived,321,2017-07-16 09:44:00+03,2017-07-16 10:39:00+03
2,PG0404,2017-08-05 19:05:00+03,2017-08-05 20:00:00+03,DME,LED,Arrived,321,2017-08-05 19:06:00+03,2017-08-05 20:01:00+03
3,PG0405,2017-08-05 09:35:00+03,2017-08-05 10:30:00+03,DME,LED,Arrived,321,2017-08-05 09:39:00+03,2017-08-05 10:34:00+03
14,PG0402,2017-08-06 12:25:00+03,2017-08-06 13:20:00+03,DME,LED,Arrived,321,2017-08-06 12:28:00+03,2017-08-06 13:23:00+03
15,PG0402,2017-07-28 12:25:00+03,2017-07-28 13:20:00+03,DME,LED,Arrived,321,2017-07-28 12:31:00+03,2017-07-28 13:26:00+03


#### Seats Dataset

In [22]:
%%sql
# load seats descriptions
SELECT *
FROM seats 
LIMIT 5;

Running query in 'sqlite:////kaggle/input/airlines-dataset/travel.sqlite'

aircraft_code,seat_no,fare_conditions
319,2A,Business
319,2C,Business
319,2D,Business
319,2F,Business
319,3A,Business


#### Ticket Flights Dataset

In [23]:
%%sql
# load ticket_flights data
SELECT *
FROM ticket_flights 
LIMIT 5;

Running query in 'sqlite:////kaggle/input/airlines-dataset/travel.sqlite'

ticket_no,flight_id,fare_conditions,amount
0005432159776,30625,Business,42100
0005435212351,30625,Business,42100
0005435212386,30625,Business,42100
0005435212381,30625,Business,42100
0005432211370,30625,Business,42100


#### Tickets Dataset

In [24]:
%%sql
# load tickets data
SELECT *
FROM tickets
LIMIT 5;

Running query in 'sqlite:////kaggle/input/airlines-dataset/travel.sqlite'

ticket_no,book_ref,passenger_id
0005432000987,06B046,8149 604011
0005432000988,06B046,8499 420203
0005432000989,E170C3,1011 752484
0005432000990,E170C3,4849 400049
0005432000991,F313DD,6615 976589


In [ ]:
%%sql
WITH flight_data AS (
       SELECT 
                f.flight_id,
                f.flight_no,
                datetime(substr(f.scheduled_departure, 1, 19)) AS scheduled_departure,
                datetime(substr(f.actual_departure, 1, 19)) AS actual_departure,
                datetime(substr(f.scheduled_arrival, 1, 19)) AS scheduled_arrival,
                datetime(substr(f.actual_arrival, 1, 19)) AS actual_arrival,
                f.departure_airport,
                f.arrival_airport,
                f.status,
                json_extract(ap1.airport_name, '$.en') AS departure_airport_name,
                json_extract(ap1.city, '$.en') AS departure_city,
                ap1.coordinates AS departure_city_coordinates,
                ap1.timezone AS departure_timezone,
                json_extract(ap2.airport_name, '$.en') AS arrival_airport_name,
                json_extract(ap2.city, '$.en') AS arrival_city,
                ap2.coordinates AS arrival_city_coordinates,
                ap2.timezone AS arrival_timezone
        FROM flights f
        JOIN airports_data ap1
        ON f.departure_airport=ap1.airport_code
        JOIN airports_data ap2
        ON f.arrival_airport=ap2.airport_code
        )
SELECT * 
FROM flight_data
WHERE actual_departure !='\\N' OR actual_arrival !='\\N'
LIMIT 10;

In [ ]:
# Top 10 Airports with the Most Delays
querry="""
WITH flight_data AS (
       SELECT 
                f.flight_id,
                f.flight_no,
                datetime(substr(f.scheduled_departure, 1, 19)) AS scheduled_departure,
                datetime(substr(f.actual_departure, 1, 19)) AS actual_departure,
                datetime(substr(f.scheduled_arrival, 1, 19)) AS scheduled_arrival,
                datetime(substr(f.actual_arrival, 1, 19)) AS actual_arrival,
                f.departure_airport,
                f.arrival_airport,
                f.status,
                json_extract(ap1.airport_name, '$.en') AS departure_airport_name,
                json_extract(ap1.city, '$.en') AS departure_city,
                ap1.coordinates AS departure_city_coordinates,
                ap1.timezone AS departure_timezone,
                json_extract(ap2.airport_name, '$.en') AS arrival_airport_name,
                json_extract(ap2.city, '$.en') AS arrival_city,
                ap2.coordinates AS arrival_city_coordinates,
                ap2.timezone AS arrival_timezone
        FROM flights f
        JOIN airports_data ap1
        ON f.departure_airport=ap1.airport_code
        JOIN airports_data ap2
        ON f.arrival_airport=ap2.airport_code
        )

SELECT departure_airport_name, 
        ROUND(AVG(strftime('%s', datetime(actual_departure)) - strftime('%s', datetime(scheduled_departure)))/60,2) AS 'avg_delay (minutes)'
FROM flight_data
WHERE actual_departure !="\\N" OR actual_arrival !="\\N"
GROUP BY departure_airport_name
ORDER BY 2 DESC
LIMIT 10;
"""
df = pd.read_sql_query(querry, sqlite_conn)

# Plot
plt.figure(figsize=(10, 5))
sns.barplot(x=df['avg_delay (minutes)'], y=df['departure_airport_name'], palette='viridis')
plt.grid(axis='x', linestyle='--', alpha=0.6)

# Add labels and title
plt.xlabel('Average Delay (minutes)')
plt.ylabel('Airport Name')
plt.title('Top 10 Airports with Highest Departure Delays')

# Show the plot
plt.show();

In [ ]:
# Flight Status Distribution
querry="""
WITH flight_data AS (
       SELECT 
                f.flight_id,
                f.flight_no,
                datetime(substr(f.scheduled_departure, 1, 19)) AS scheduled_departure,
                datetime(substr(f.actual_departure, 1, 19)) AS actual_departure,
                datetime(substr(f.scheduled_arrival, 1, 19)) AS scheduled_arrival,
                datetime(substr(f.actual_arrival, 1, 19)) AS actual_arrival,
                f.departure_airport,
                f.arrival_airport,
                f.status,
                json_extract(ap1.airport_name, '$.en') AS departure_airport_name,
                json_extract(ap1.city, '$.en') AS departure_city,
                ap1.coordinates AS departure_city_coordinates,
                ap1.timezone AS departure_timezone,
                json_extract(ap2.airport_name, '$.en') AS arrival_airport_name,
                json_extract(ap2.city, '$.en') AS arrival_city,
                ap2.coordinates AS arrival_city_coordinates,
                ap2.timezone AS arrival_timezone
        FROM flights f
        JOIN airports_data ap1
        ON f.departure_airport=ap1.airport_code
        JOIN airports_data ap2
        ON f.arrival_airport=ap2.airport_code
        )

SELECT status, 
        COUNT(*) AS Counts
FROM flight_data
GROUP BY status
ORDER BY 2 DESC;
"""
df=pd.read_sql_query(querry, sqlite_conn)
plt.figure(figsize=(6, 6))
plt.pie(df['Counts'], labels=df['status'], 
        autopct='%1.1f%%', startangle=30, 
        colors=sns.color_palette('viridis', 5),
        wedgeprops=dict(width=0.6, edgecolor='w'),
        textprops={'fontsize': 11},
        shadow=True,
        pctdistance=0.85)

plt.title('Flight Status Distribution')
plt.show();

In [ ]:
# Top 10 Airports with the Most Delays
querry="""
WITH flight_data AS (
       SELECT 
                f.flight_id,
                f.flight_no,
                datetime(substr(f.scheduled_departure, 1, 19)) AS scheduled_departure,
                datetime(substr(f.actual_departure, 1, 19)) AS actual_departure,
                datetime(substr(f.scheduled_arrival, 1, 19)) AS scheduled_arrival,
                datetime(substr(f.actual_arrival, 1, 19)) AS actual_arrival,
                f.departure_airport,
                f.arrival_airport,
                f.status,
                json_extract(ap1.airport_name, '$.en') AS departure_airport_name,
                json_extract(ap1.city, '$.en') AS departure_city,
                ap1.coordinates AS departure_city_coordinates,
                ap1.timezone AS departure_timezone,
                json_extract(ap2.airport_name, '$.en') AS arrival_airport_name,
                json_extract(ap2.city, '$.en') AS arrival_city,
                ap2.coordinates AS arrival_city_coordinates,
                ap2.timezone AS arrival_timezone
        FROM flights f
        JOIN airports_data ap1
        ON f.departure_airport=ap1.airport_code
        JOIN airports_data ap2
        ON f.arrival_airport=ap2.airport_code
        )

SELECT flight_no, 
        ROUND(AVG(strftime('%s', datetime(actual_departure)) - strftime('%s', datetime(scheduled_departure)))/60,2) AS 'avg_delay (minutes)'
FROM flight_data
WHERE actual_departure !="\\N" OR actual_arrival !="\\N"
GROUP BY flight_no 
ORDER BY 2 DESC
LIMIT 10;
"""
dff=pd.read_sql_query(querry, sqlite_conn)
dff

In [ ]:
%%sql
# Most Popular Flight Hours
WITH flight_data AS (
       SELECT 
                f.flight_id,
                f.flight_no,
                datetime(substr(f.scheduled_departure, 1, 19)) AS scheduled_departure,
                datetime(substr(f.actual_departure, 1, 19)) AS actual_departure,
                datetime(substr(f.scheduled_arrival, 1, 19)) AS scheduled_arrival,
                datetime(substr(f.actual_arrival, 1, 19)) AS actual_arrival,
                f.departure_airport,
                f.arrival_airport,
                f.status,
                json_extract(ap1.airport_name, '$.en') AS departure_airport_name,
                json_extract(ap1.city, '$.en') AS departure_city,
                ap1.coordinates AS departure_city_coordinates,
                ap1.timezone AS departure_timezone,
                json_extract(ap2.airport_name, '$.en') AS arrival_airport_name,
                json_extract(ap2.city, '$.en') AS arrival_city,
                ap2.coordinates AS arrival_city_coordinates,
                ap2.timezone AS arrival_timezone
        FROM flights f
        JOIN airports_data ap1
        ON f.departure_airport=ap1.airport_code
        JOIN airports_data ap2
        ON f.arrival_airport=ap2.airport_code
        )

SELECT strftime('%HH', datetime(scheduled_departure)) AS departure_hour, 
       COUNT(*) AS flight_count
FROM flight_data
WHERE actual_departure !="\\N" OR actual_arrival !="\\N"
GROUP BY departure_hour
ORDER BY 2 DESC
LIMIT 10;

In [ ]:
# Most Popular Flight Hours
querry="""
WITH flight_data AS (
       SELECT 
                f.flight_id,
                f.flight_no,
                datetime(substr(f.scheduled_departure, 1, 19)) AS scheduled_departure,
                datetime(substr(f.actual_departure, 1, 19)) AS actual_departure,
                datetime(substr(f.scheduled_arrival, 1, 19)) AS scheduled_arrival,
                datetime(substr(f.actual_arrival, 1, 19)) AS actual_arrival,
                f.departure_airport,
                f.arrival_airport,
                f.status,
                json_extract(ap1.airport_name, '$.en') AS departure_airport_name,
                json_extract(ap1.city, '$.en') AS departure_city,
                ap1.coordinates AS departure_city_coordinates,
                ap1.timezone AS departure_timezone,
                json_extract(ap2.airport_name, '$.en') AS arrival_airport_name,
                json_extract(ap2.city, '$.en') AS arrival_city,
                ap2.coordinates AS arrival_city_coordinates,
                ap2.timezone AS arrival_timezone
        FROM flights f
        JOIN airports_data ap1
        ON f.departure_airport=ap1.airport_code
        JOIN airports_data ap2
        ON f.arrival_airport=ap2.airport_code
        )

SELECT strftime('%HH', datetime(scheduled_departure)) AS departure_hour, 
       COUNT(*) AS flight_count
FROM flight_data
WHERE actual_departure !="\\N" OR actual_arrival !="\\N"
GROUP BY departure_hour
ORDER BY 2 DESC
LIMIT 10;
"""
df_hour=pd.read_sql_query(querry, sqlite_conn)
df_hour


In [ ]:
%%sql
# Peak Travel Days
WITH flight_data AS (
       SELECT 
                f.flight_id,
                f.flight_no,
                datetime(substr(f.scheduled_departure, 1, 19)) AS scheduled_departure,
                datetime(substr(f.actual_departure, 1, 19)) AS actual_departure,
                datetime(substr(f.scheduled_arrival, 1, 19)) AS scheduled_arrival,
                datetime(substr(f.actual_arrival, 1, 19)) AS actual_arrival,
                f.departure_airport,
                f.arrival_airport,
                f.status,
                json_extract(ap1.airport_name, '$.en') AS departure_airport_name,
                json_extract(ap1.city, '$.en') AS departure_city,
                ap1.coordinates AS departure_city_coordinates,
                ap1.timezone AS departure_timezone,
                json_extract(ap2.airport_name, '$.en') AS arrival_airport_name,
                json_extract(ap2.city, '$.en') AS arrival_city,
                ap2.coordinates AS arrival_city_coordinates,
                ap2.timezone AS arrival_timezone
        FROM flights f
        JOIN airports_data ap1
        ON f.departure_airport=ap1.airport_code
        JOIN airports_data ap2
        ON f.arrival_airport=ap2.airport_code
        )

SELECT CASE strftime('%w', scheduled_departure)
        WHEN '0' THEN 'Sunday'
        WHEN '1' THEN 'Monday'
        WHEN '2' THEN 'Tuesday'
        WHEN '3' THEN 'Wednesday'
        WHEN '4' THEN 'Thursday'
        WHEN '5' THEN 'Friday'
        WHEN '6' THEN 'Saturday'
    END AS flight_weekday,
       COUNT(*) AS total_flights
FROM flight_data
WHERE actual_departure !="\\N" OR actual_arrival !="\\N"
GROUP BY flight_weekday
ORDER BY 2 DESC
LIMIT 10;

In [ ]:
# Peak Travel Days
querry="""
WITH flight_data AS (
       SELECT 
                f.flight_id,
                f.flight_no,
                datetime(substr(f.scheduled_departure, 1, 19)) AS scheduled_departure,
                datetime(substr(f.actual_departure, 1, 19)) AS actual_departure,
                datetime(substr(f.scheduled_arrival, 1, 19)) AS scheduled_arrival,
                datetime(substr(f.actual_arrival, 1, 19)) AS actual_arrival,
                f.departure_airport,
                f.arrival_airport,
                f.status,
                json_extract(ap1.airport_name, '$.en') AS departure_airport_name,
                json_extract(ap1.city, '$.en') AS departure_city,
                ap1.coordinates AS departure_city_coordinates,
                ap1.timezone AS departure_timezone,
                json_extract(ap2.airport_name, '$.en') AS arrival_airport_name,
                json_extract(ap2.city, '$.en') AS arrival_city,
                ap2.coordinates AS arrival_city_coordinates,
                ap2.timezone AS arrival_timezone
        FROM flights f
        JOIN airports_data ap1
        ON f.departure_airport=ap1.airport_code
        JOIN airports_data ap2
        ON f.arrival_airport=ap2.airport_code
        )

SELECT CASE strftime('%w', scheduled_departure)
        WHEN '0' THEN 'Sunday'
        WHEN '1' THEN 'Monday'
        WHEN '2' THEN 'Tuesday'
        WHEN '3' THEN 'Wednesday'
        WHEN '4' THEN 'Thursday'
        WHEN '5' THEN 'Friday'
        WHEN '6' THEN 'Saturday'
    END AS flight_weekday,
       COUNT(*) AS total_flights
FROM flight_data
WHERE actual_departure !="\\N" OR actual_arrival !="\\N"
GROUP BY flight_weekday
ORDER BY 2 DESC
LIMIT 10;
"""
df_days=pd.read_sql_query(querry, sqlite_conn)
df_days